# E16 生存 / 流失：Kaplan–Meier 与删失

> 覆盖矩阵：生存分析

## 你将学到
1. **删失**：观测结束时事件未发生，不能当「没流失」直接分类。
2. Kaplan–Meier 曲线与两组 log-rank 思想。
3. 与普通二分类流失模型的边界。
4. **缺口**：无 `SurvivalAnalyzer`；可选 extras `[survival]`。


In [ ]:
QUICK = True
RANDOM_STATE = 42
from wanglaoshi import ready
ready(seed=RANDOM_STATE)


## 1. 合成订阅流失数据

In [ ]:
import numpy as np
import pandas as pd
from wanglaoshi import DataAnalyzer

rng = np.random.default_rng(RANDOM_STATE)
n = 600
plan = rng.choice(['basic', 'pro'], n, p=[0.6, 0.4])
# pro 流失更慢
scale = np.where(plan == 'pro', 18.0, 10.0)
tenure = rng.exponential(scale)
censor_time = 24.0
event = (tenure <= censor_time).astype(int)
time = np.minimum(tenure, censor_time)
df = pd.DataFrame({'plan': plan, 'time': time, 'event': event})
print(df.groupby('plan')[['time', 'event']].mean())
print(DataAnalyzer(df).get_basic_info().get('shape'))


## 2. 手写 KM（无额外依赖）

In [ ]:
def kaplan_meier(times, events):
    order = np.argsort(times)
    times, events = times[order], events[order]
    uniq = np.unique(times)
    surv, t_axis = [1.0], [0.0]
    at_risk = len(times)
    s = 1.0
    for t in uniq:
        died = events[times == t].sum()
        if at_risk > 0 and died > 0:
            s *= (1 - died / at_risk)
        surv.append(s)
        t_axis.append(t)
        at_risk -= np.sum(times == t)
    return np.array(t_axis), np.array(surv)

import matplotlib.pyplot as plt
from pathlib import Path
out = Path('outputs/16_survival_churn'); out.mkdir(parents=True, exist_ok=True)
plt.figure(figsize=(7, 4))
for name, g in df.groupby('plan'):
    t_axis, surv = kaplan_meier(g['time'].to_numpy(), g['event'].to_numpy())
    plt.step(t_axis, surv, where='post', label=name)
plt.xlabel('time'); plt.ylabel('S(t)'); plt.legend(); plt.title('KM by plan')
plt.tight_layout(); plt.savefig(out / 'km.png', dpi=100); plt.close()
print('saved km.png')


## 3. 简易 log-rank（两组）

In [ ]:
# 使用 lifelines 若可用，否则跳过并打印提示
try:
    from lifelines import KaplanMeierFitter
    from lifelines.statistics import logrank_test
    a = df[df.plan == 'basic']
    b = df[df.plan == 'pro']
    lr = logrank_test(a['time'], b['time'], event_observed_A=a['event'], event_observed_B=b['event'])
    print('log-rank p=', lr.p_value)
except Exception as e:
    print('lifelines 可选未装，跳过 log-rank；手写 KM 已足够跑通 evidence。', type(e).__name__)


## 4. 错误做法对照：忽略删失直接分类

In [ ]:
# 错误：把 event 当普通标签、忽略 time —— 偏倚风险
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
X = LabelEncoder().fit_transform(df['plan']).reshape(-1, 1)
y = df['event']
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y)
clf = LogisticRegression().fit(Xtr, ytr)
print('naive accuracy', clf.score(Xte, yte), '（教学：指标看起来「能跑」但问题形式错了）')
pd.DataFrame({'n': [len(df)], 'event_rate': [df.event.mean()]}).to_csv(out / 'survival_summary.csv', index=False)


## 附录
- 作业：加入协变量做 Cox（需 lifelines）；对比仅用 `event` 分类的偏差叙述。
- 包优化：`SurvivalAnalyzer.km_plot` / `logrank`，依赖放 extras。
